In [1]:
from pathlib import Path
import mne
import numpy as np
from fooof import FOOOF

/var/folders/rh/p5yby44s3_q7gw0x4t4155p40000gq/T/ipykernel_8855/3248309884.py:4: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof import FOOOF


In [2]:
import argparse
import itertools
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic, gen_periodic

In [3]:
DATA_FOLDER1 = Path("DATA/posterior")
DATA_FOLDER2 = Path("DATA/frontal")

EPOCH_FILES1 = sorted(DATA_FOLDER1.glob("*-epo.fif"))
EPOCH_FILES2 = sorted(DATA_FOLDER2.glob("*-epo.fif"))

In [ ]:
"""
validate_iaf.py - sanity checks for FOOOF-based individual alpha frequency (IAF).

What it does
------------
1. Epoch bookkeeping: number of epochs, epoch length, frequency resolution per file.
2. Fit QC: R^2, error, number of peaks; outliers flagged (> 2.5 SD within region,
   as in Donoghue et al. 2020).
3. Method agreement: FOOOF IAF vs. raw-PSD argmax, flattened-spectrum argmax and
   centre of gravity in the alpha window.
4. Sensitivity: refits over a grid of FOOOF settings (and mean vs. median epoch
   averaging) and reports how much the IAF moves.
5. Simulation: synthetic spectra with KNOWN peaks, built on the frequency axis and
   aperiodic/noise statistics of YOUR real fits. Compares your original settings
   with the primary (paper-like) settings: IAF error, detection rate, false-positive
   rate when there is no peak.
6. Diagnostic plots (log power + model fit) for flagged spectra and a random sample.

Run:   python validate_iaf.py --data-dir DATA --out-dir IAF_validation
Notebook:  %run validate_iaf.py   (or call main(["--data-dir", "DATA"]))

Requires fooof 1.x (pip install fooof), mne, numpy, pandas, matplotlib.
"""
import argparse
import itertools
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic, gen_periodic

# ----------------------------------------------------------------------------------
# CONFIGURATION (edit to match your pipeline)
# ----------------------------------------------------------------------------------
FREQ_MIN, FREQ_MAX = 2.0, 40.0
ALPHA_MIN, ALPHA_MAX = 8.0, 14.0

REGIONS = {
    "frontal": ["AFz", "Fz", "FCz", "F1", "F2", "FC1", "FC2"],
    "posterior": ["CPz", "Pz", "POz", "P1", "P2"],
}

# Settings you used so far (kept as-is so the comparison is fair)
ORIGINAL = dict(peak_width_limits=(1.0, 2.5), max_n_peaks=8, min_peak_height=0.0,
                peak_threshold=0.5, aperiodic_mode="fixed")

# Paper-like settings. Lower width bound is set at run time to >= 2x freq resolution.
PRIMARY_BASE = dict(max_n_peaks=6, min_peak_height=0.05, peak_threshold=1.5,
                    aperiodic_mode="fixed")
PRIMARY_UPPER_WIDTH = 6.0

# 2nd settings
ORIGINAL = dict(peak_width_limits=(2.0, 3.0), max_n_peaks=8, min_peak_height=0.0,
                peak_threshold=0.5, aperiodic_mode="fixed")

# Sensitivity grid
GRID_UPPER_WIDTH = [2.5, 6.0, 12.0]
GRID_THRESHOLD = [1.0, 1.5, 2.0]
GRID_MIN_HEIGHT = [0.0, 0.05, 0.1]
GRID_AGG = ["mean", "median"]          # how epochs are averaged

# Flag thresholds
MIN_EPOCHS = 30            # rule of thumb, not a hard rule
OUTLIER_SD = 2.5           # paper: R^2 / error > 2.5 SD from the mean
IAF_RANGE_TOL = 0.5        # Hz: IAF spread across the sensitivity grid
SIM_TRUE_ERR_TOL = 0.5     # Hz: counts as "accurate" in simulations


# ----------------------------------------------------------------------------------
# DATA HELPERS (same logic as your pipeline)
# ----------------------------------------------------------------------------------
def get_subject_id(file_path):
    name = file_path.name
    for suffix in ("-epo.fif", "_epo.fif"):
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    if name.endswith("_preprocessed"):
        name = name[:-len("_preprocessed")]
    return name


def roi_psds(epochs, region_channels):
    """Per-epoch, per-channel Welch PSD for the ROI: (n_epochs, n_ch, n_freqs)."""
    available = [ch for ch in region_channels
                 if ch in epochs.ch_names and ch not in epochs.info["bads"]]
    if not available:
        return None, None, []
    roi = epochs.copy().pick(available)
    sfreq = roi.info["sfreq"]
    n_times = len(roi.times)
    n_per_seg = min(round(sfreq * 2), n_times)
    # one segment per epoch if the epoch is shorter than 2 s -> no overlap possible
    n_overlap = 0 if n_per_seg == n_times else round(sfreq)
    spec = roi.compute_psd(method="welch", fmin=FREQ_MIN, fmax=FREQ_MAX,
                           n_per_seg=n_per_seg, n_overlap=n_overlap,
                           average="mean", verbose=False)
    psds, freqs = spec.get_data(return_freqs=True)
    return freqs, psds, available


def aggregate(psds, how):
    per_epoch = psds.mean(axis=1)                      # mean across channels
    return per_epoch.mean(axis=0) if how == "mean" else np.median(per_epoch, axis=0)


# ----------------------------------------------------------------------------------
# FITTING + FEATURES
# ----------------------------------------------------------------------------------
def primary_settings(freq_res):
    lower = max(1.0, 2 * freq_res)
    return dict(PRIMARY_BASE, peak_width_limits=(lower, PRIMARY_UPPER_WIDTH))


def fit_spectrum(freqs, psd, settings):
    fm = FOOOF(**settings, verbose=False)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            fm.fit(freqs, psd, [FREQ_MIN, FREQ_MAX])
        except Exception:
            pass
    return fm


def fit_features(fm):
    """IAF = centre frequency of the highest peak inside the alpha window (else NaN)."""
    out = dict(iaf=np.nan, alpha_height=np.nan, alpha_bw=np.nan, n_alpha_peaks=0,
               n_peaks=0, r2=np.nan, error=np.nan, offset=np.nan, exponent=np.nan,
               resid_sd=np.nan)
    ap = np.asarray(fm.aperiodic_params_, dtype=float)
    if ap.size < 2 or np.isnan(ap).any():
        return out
    pk = np.atleast_2d(fm.peak_params_)
    pk = pk[~np.isnan(pk).any(axis=1)] if pk.size else np.empty((0, 3))
    in_alpha = pk[(pk[:, 0] >= ALPHA_MIN) & (pk[:, 0] <= ALPHA_MAX)] if len(pk) else pk
    if len(in_alpha):
        best = in_alpha[np.argmax(in_alpha[:, 1])]
        out.update(iaf=float(best[0]), alpha_height=float(best[1]), alpha_bw=float(best[2]))
    out.update(n_alpha_peaks=len(in_alpha), n_peaks=len(pk), r2=float(fm.r_squared_),
               error=float(fm.error_), offset=float(ap[0]), exponent=float(ap[1]))
    try:
        out["resid_sd"] = float(np.std(fm.power_spectrum - fm.fooofed_spectrum_))
    except Exception:
        pass
    return out


def alt_methods(fm, freqs, psd):
    """Model-free / semi-model-free IAF estimates for cross-checking."""
    res = dict(iaf_raw=np.nan, raw_at_edge=False, iaf_flat=np.nan, flat_at_edge=False,
               flat_height=np.nan, iaf_cog=np.nan)
    m = (freqs >= ALPHA_MIN) & (freqs <= ALPHA_MAX)
    if m.any():
        idx = int(np.argmax(psd[m]))
        res["iaf_raw"] = float(freqs[m][idx])
        res["raw_at_edge"] = idx in (0, m.sum() - 1)
    ap = np.asarray(fm.aperiodic_params_, dtype=float)
    if ap.size < 2 or np.isnan(ap).any():
        return res
    flat = fm.power_spectrum - gen_aperiodic(fm.freqs, ap)
    mf = (fm.freqs >= ALPHA_MIN) & (fm.freqs <= ALPHA_MAX)
    if mf.any():
        idx = int(np.argmax(flat[mf]))
        res["iaf_flat"] = float(fm.freqs[mf][idx])
        res["flat_at_edge"] = idx in (0, mf.sum() - 1)
        res["flat_height"] = float(flat[mf][idx])
        pos = np.clip(flat[mf], 0, None)
        if pos.sum() > 0:
            res["iaf_cog"] = float(np.sum(fm.freqs[mf] * pos) / pos.sum())
    return res


def run_sensitivity(freqs, psds_by_agg, freq_res):
    lower = max(1.0, 2 * freq_res)
    iafs, r2s = [], []
    for agg, psd in psds_by_agg.items():
        for upper, thr, hgt in itertools.product(GRID_UPPER_WIDTH, GRID_THRESHOLD,
                                                 GRID_MIN_HEIGHT):
            s = dict(PRIMARY_BASE, peak_width_limits=(lower, upper),
                     peak_threshold=thr, min_peak_height=hgt)
            f = fit_features(fit_spectrum(freqs, psd, s))
            iafs.append(f["iaf"])
            r2s.append(f["r2"])
    iafs = np.array(iafs, dtype=float)
    found = ~np.isnan(iafs)
    return dict(
        sens_n_configs=len(iafs),
        sens_share_alpha=float(found.mean()),
        sens_iaf_median=float(np.nanmedian(iafs)) if found.any() else np.nan,
        sens_iaf_range=float(np.nanmax(iafs) - np.nanmin(iafs)) if found.any() else np.nan,
        sens_iaf_sd=float(np.nanstd(iafs)) if found.any() else np.nan,
        sens_r2_min=float(np.nanmin(r2s)),
    )


# ----------------------------------------------------------------------------------
# PLOTS
# ----------------------------------------------------------------------------------
def plot_qc(fm, psd_raw, freqs_raw, row, out_path):
    ap = np.asarray(fm.aperiodic_params_, dtype=float)
    if ap.size < 2 or np.isnan(ap).any():
        return
    ap_fit = gen_aperiodic(fm.freqs, ap)
    flat = fm.power_spectrum - ap_fit
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    ax = axes[0]
    ax.plot(fm.freqs, fm.power_spectrum, color="k", lw=1.4, label="PSD (log10)")
    ax.plot(fm.freqs, fm.fooofed_spectrum_, color="red", lw=1.4, label="full model")
    ax.plot(fm.freqs, ap_fit, color="blue", ls="--", lw=1.2, label="aperiodic fit")
    ax.axvspan(ALPHA_MIN, ALPHA_MAX, color="grey", alpha=0.12)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("log10 power")
    ax.legend(frameon=False, fontsize=8)

    ax = axes[1]
    ax.plot(fm.freqs, flat, color="k", lw=1.2, label="PSD - aperiodic")
    ax.plot(fm.freqs, fm.fooofed_spectrum_ - ap_fit, color="red", lw=1.2, label="fitted peaks")
    ax.axhline(0, color="grey", lw=0.6)
    ax.axvspan(ALPHA_MIN, ALPHA_MAX, color="grey", alpha=0.12)
    marks = [("iaf", "red", "-", "FOOOF"), ("iaf_flat", "green", ":", "flat argmax"),
             ("iaf_raw", "darkorange", ":", "raw argmax"), ("iaf_cog", "purple", ":", "CoG")]
    for key, c, ls, lab in marks:
        v = row.get(key, np.nan)
        if np.isfinite(v):
            ax.axvline(v, color=c, ls=ls, lw=1.4, label=f"{lab} {v:.2f}")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("flattened log10 power")
    ax.legend(frameon=False, fontsize=8)

    flags = row.get("flags_str", "")
    fig.suptitle(f"{row['subject_id']} | {row['region']} | R2={row['r2']:.3f} "
                 f"err={row['error']:.3f} peaks={row['n_peaks']} | {flags or 'no flags'}",
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=130)
    plt.close(fig)


# ----------------------------------------------------------------------------------
# SIMULATION
# ----------------------------------------------------------------------------------
def make_spectrum(freqs, ap_params, peaks, nlv, rng):
    log_p = gen_aperiodic(freqs, np.asarray(ap_params))
    if peaks:
        log_p = log_p + gen_periodic(freqs, np.ravel(peaks))
    log_p = log_p + rng.normal(0, nlv, size=freqs.size)
    return 10 ** log_p


def simulate_recovery(freqs, real_df, settings_by_name, n_sim, rng):
    """Spectra with known truth, on your frequency axis and your aperiodic/noise stats."""
    ok = real_df.dropna(subset=["offset", "exponent", "resid_sd"])
    if ok.empty:
        return pd.DataFrame()
    noise_levels = dict(zip(["low", "median", "high"],
                            np.quantile(ok["resid_sd"], [0.25, 0.5, 0.9])))
    cases = {
        "alpha_strong": (0.30, 0.50, False),
        "alpha_medium": (0.15, 0.30, False),
        "alpha_weak": (0.05, 0.15, False),
        "alpha_plus_beta": (0.15, 0.30, True),
        "no_peak": (None, None, False),
    }
    rows = []
    for case, (h_lo, h_hi, add_beta) in cases.items():
        for nl_name, nlv in noise_levels.items():
            sims = []
            for _ in range(n_sim):
                r = ok.iloc[rng.integers(len(ok))]
                ap = [r["offset"], r["exponent"]]
                peaks, true_cf = [], np.nan
                if h_lo is not None:
                    true_cf = rng.uniform(8.5, 13.0)
                    sd = rng.uniform(0.5, 1.5)           # bandwidth 1-3 Hz (2 SD)
                    peaks.append([true_cf, rng.uniform(h_lo, h_hi), sd])
                    if add_beta:
                        peaks.append([rng.uniform(18, 25), rng.uniform(0.05, 0.2),
                                      rng.uniform(1.0, 2.5)])
                sims.append((make_spectrum(freqs, ap, peaks, nlv, rng), true_cf))
            for sname, settings in settings_by_name.items():
                for spec, true_cf in sims:
                    f = fit_features(fit_spectrum(freqs, spec, settings))
                    detected = np.isfinite(f["iaf"])
                    rows.append(dict(
                        settings=sname, case=case, noise=nl_name, noise_sd=nlv,
                        true_cf=true_cf, est_iaf=f["iaf"], detected=detected,
                        abs_err=abs(f["iaf"] - true_cf) if detected and np.isfinite(true_cf) else np.nan,
                        n_alpha_peaks=f["n_alpha_peaks"], r2=f["r2"]))
    return pd.DataFrame(rows)


def summarise_simulation(sim_df):
    rows = []
    for (case, noise, sname), g in sim_df.groupby(["case", "noise", "settings"]):
        out = dict(case=case, noise=noise, settings=sname, n=len(g),
                   detection_rate=g["detected"].mean(),
                   mean_alpha_peaks=g["n_alpha_peaks"].mean())
        if g["true_cf"].notna().any():
            e = g["abs_err"].dropna()
            out.update(median_abs_err=e.median() if len(e) else np.nan,
                       p95_abs_err=e.quantile(0.95) if len(e) else np.nan,
                       share_err_le_tol=(e <= SIM_TRUE_ERR_TOL).sum() / len(g))
        else:
            out.update(false_positive_rate=g["detected"].mean())
        rows.append(out)
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------------
def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--data-dir", default="DATA")
    p.add_argument("--out-dir", default="IAF_validation")
    p.add_argument("--n-sim", type=int, default=100, help="simulations per cell")
    p.add_argument("--n-plots", type=int, default=10, help="random unflagged plots per region")
    p.add_argument("--skip-sim", action="store_true")
    p.add_argument("--seed", type=int, default=0)
    # parse_known_args: inside Jupyter/VS Code, sys.argv holds kernel arguments (--f=...)
    args, _unknown = p.parse_known_args(argv)

    rng = np.random.default_rng(args.seed)
    data_dir, out_dir = Path(args.data_dir), Path(args.out_dir)
    plots_dir = out_dir / "qc_plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(set(data_dir.rglob("*-epo.fif")) | set(data_dir.rglob("*_epo.fif")))
    if not files:
        raise FileNotFoundError(f"No epoch files in {data_dir.resolve()}")

    rows, store = [], {}
    ref_freqs = None

    for fp in files:
        region = fp.parent.name.lower()
        if region not in REGIONS:
            continue
        sid = get_subject_id(fp)
        epochs = mne.read_epochs(fp, preload=True, verbose=False)
        freqs, psds, used = roi_psds(epochs, REGIONS[region])
        if freqs is None:
            print(f"skip {sid} ({region}): no usable ROI channels")
            continue
        ref_freqs = freqs if ref_freqs is None else ref_freqs
        n_times = len(epochs.times)
        sfreq = epochs.info["sfreq"]
        seg_len = min(2.0, n_times / sfreq)
        freq_res = float(np.mean(np.diff(freqs)))

        psd_by_agg = {a: aggregate(psds, a) for a in GRID_AGG}
        psd = psd_by_agg["mean"]                       # what your pipeline used

        fm = fit_spectrum(freqs, psd, primary_settings(freq_res))
        feat = fit_features(fm)
        alt = alt_methods(fm, freqs, psd)
        sens = run_sensitivity(freqs, psd_by_agg, freq_res)

        fm_med = fit_spectrum(freqs, psd_by_agg["median"], primary_settings(freq_res))
        iaf_median_agg = fit_features(fm_med)["iaf"]
        iaf_orig = fit_features(fit_spectrum(freqs, psd, ORIGINAL))["iaf"]

        row = dict(subject_id=sid, region=region, n_epochs=len(epochs),
                   n_channels=len(used), epoch_len_s=n_times / sfreq,
                   welch_seg_s=seg_len, freq_res_hz=freq_res, **feat, **alt, **sens,
                   iaf_median_epoch_agg=iaf_median_agg, iaf_original_settings=iaf_orig)
        rows.append(row)
        store[(sid, region)] = (fm, psd, freqs)
        print(f"{sid:>12} {region:>9} | epochs={len(epochs):3d} | IAF={feat['iaf']:.2f} "
              f"| R2={feat['r2']:.3f} | peaks={feat['n_peaks']}")

    if not rows:
        raise RuntimeError("Nothing processed - check folder names (frontal/posterior).")

    df = pd.DataFrame(rows)

    # ---------------- flags ----------------
    for col in ("r2", "error"):
        g = df.groupby("region")[col]
        df[f"z_{col}"] = (df[col] - g.transform("mean")) / g.transform("std").replace(0, np.nan)

    tol = np.maximum(IAF_RANGE_TOL, df["freq_res_hz"])
    df["flag_no_alpha_peak"] = df["iaf"].isna()
    df["flag_bad_fit"] = (df["z_r2"] < -OUTLIER_SD) | (df["z_error"] > OUTLIER_SD) | df["r2"].isna()
    df["flag_max_peaks"] = df["n_peaks"] >= PRIMARY_BASE["max_n_peaks"]
    df["flag_multi_alpha_peaks"] = df["n_alpha_peaks"] > 1
    df["flag_method_disagree"] = (
        ((df["iaf"] - df["iaf_flat"]).abs() > tol)
        | ((df["iaf"] - df["iaf_raw"]).abs() > tol)
        | ((df["iaf"] - df["iaf_cog"]).abs() > 2 * tol))   # CoG is broader -> looser
    df["flag_unstable_settings"] = (
        (df["sens_iaf_range"] > IAF_RANGE_TOL)
        | ((df["sens_share_alpha"] > 0) & (df["sens_share_alpha"] < 1)))
    df["flag_epoch_agg_differs"] = (df["iaf"] - df["iaf_median_epoch_agg"]).abs() > tol
    df["flag_few_epochs"] = df["n_epochs"] < MIN_EPOCHS
    df["flag_raw_at_edge"] = df["raw_at_edge"] & df["iaf"].notna()
    df["changed_vs_original"] = (df["iaf"] - df["iaf_original_settings"]).abs() > tol

    review_cols = ["flag_bad_fit", "flag_max_peaks", "flag_method_disagree",
                   "flag_unstable_settings", "flag_epoch_agg_differs", "flag_raw_at_edge"]
    df["needs_review"] = df[review_cols].any(axis=1)
    flag_cols = [c for c in df.columns if c.startswith("flag_")]
    df["flags_str"] = df[flag_cols].apply(
        lambda r: ",".join(c.replace("flag_", "") for c in flag_cols if r[c]), axis=1)

    df.to_csv(out_dir / "iaf_validation_summary.csv", index=False)

    # ---------------- plots ----------------
    review = df[df["needs_review"]]
    ok_df = df[~df["needs_review"] & df["iaf"].notna()]
    sample = pd.concat([g.sample(min(args.n_plots, len(g)), random_state=args.seed)
                        for _, g in ok_df.groupby("region")]) if len(ok_df) else ok_df
    for _, r in pd.concat([review, sample]).iterrows():
        fm, psd, freqs = store[(r["subject_id"], r["region"])]
        prefix = "REVIEW_" if r["needs_review"] else "ok_"
        plot_qc(fm, psd, freqs, r.to_dict(),
                plots_dir / f"{prefix}{r['subject_id']}_{r['region']}.png")

    # ---------------- simulation ----------------
    sim_summary = None
    if not args.skip_sim:
        freq_res = float(df["freq_res_hz"].median())
        settings = {"original": ORIGINAL, "primary": primary_settings(freq_res)}
        print("\nRunning simulations ...")
        sim_df = simulate_recovery(ref_freqs, df, settings, args.n_sim, rng)
        sim_df.to_csv(out_dir / "iaf_simulation_raw.csv", index=False)
        sim_summary = summarise_simulation(sim_df)
        sim_summary.to_csv(out_dir / "iaf_simulation_summary.csv", index=False)

    # ---------------- report ----------------
    pd.set_option("display.width", 200, "display.max_columns", 30)
    print("\n================ REPORT ================")
    print("\nEpochs per spectrum (by region):")
    print(df.groupby("region")["n_epochs"].describe()[["count", "min", "50%", "max"]])
    print(f"\nEpoch length: {sorted(df['epoch_len_s'].round(2).unique())} s | "
          f"Welch segment: {sorted(df['welch_seg_s'].round(2).unique())} s | "
          f"freq resolution: {sorted(df['freq_res_hz'].round(3).unique())} Hz")
    print("\nFit quality (primary settings):")
    print(df.groupby("region")[["r2", "error", "n_peaks"]].agg(["median", "min", "max"]).round(3))
    print("\nFlag counts (n spectra):")
    print(df.groupby("region")[flag_cols + ["needs_review", "changed_vs_original"]].sum().T)
    print(f"\nSpectra needing review: {int(df['needs_review'].sum())} of {len(df)} "
          f"-> see {plots_dir}/REVIEW_*.png")
    if sim_summary is not None:
        print("\nSimulation (known truth), original vs primary settings:")
        print(sim_summary.round(3).to_string(index=False))
    print(f"\nAll outputs in: {out_dir.resolve()}")
    return df


if __name__ == "__main__":
    main()

 HP1505_M_23   frontal | epochs= 65 | IAF=9.46 | R2=0.994 | peaks=6
 HZ1304_F_29   frontal | epochs= 73 | IAF=11.69 | R2=0.994 | peaks=6
 IZ2310_F_21   frontal | epochs= 75 | IAF=11.08 | R2=0.982 | peaks=6
 JA0310_F_20   frontal | epochs= 74 | IAF=10.58 | R2=0.987 | peaks=6
 JB0612_F_20   frontal | epochs= 72 | IAF=10.90 | R2=0.988 | peaks=5
 HP1505_M_23 posterior | epochs= 66 | IAF=9.43 | R2=0.990 | peaks=5
 HZ1304_F_29 posterior | epochs= 75 | IAF=13.38 | R2=0.996 | peaks=6
 IZ2310_F_21 posterior | epochs= 70 | IAF=11.03 | R2=0.993 | peaks=6
 JA0310_F_20 posterior | epochs= 75 | IAF=11.12 | R2=0.989 | peaks=6
 JB0612_F_20 posterior | epochs= 71 | IAF=10.72 | R2=0.987 | peaks=6

Running simulations ...

================ REPORT ================

Epochs per spectrum (by region):
           count   min   50%   max
region                            
frontal      5.0  65.0  73.0  75.0
posterior    5.0  66.0  71.0  75.0

Epoch length: [np.float64(1.8)] s | Welch segment: [np.float64(1.8)] s